#  **Data Collection and Preprocessing**

### Import Libraries

In [1]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

In [2]:
import pandas as pd
import numpy as np
from google_play_scraper import app, Sort, reviews
import re
from datetime import datetime
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
)
from src.data_scrapping import scrap_reviews

### Web Scraping

#### App metadata

In [3]:
DAHSEN_APP_ID = 'com.dashen.dashensuperapp'
display_app_info(DAHSEN_APP_ID)

Dashen Bank App Info
App Title   : Dashen Bank
Current Score: 4.22913
Total Ratings: 5,618
Total Reviews: 1,023
Installs     : 1,000,000+


#### Scrape reviews

In [4]:
reviews = scrap_reviews(app_id=DAHSEN_APP_ID)

Scraping reviews for com.dashen.dashensuperapp...
Collected 700 raw reviews


#### Collect review text, rating, review date, bank , source

In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 88c562ed-6c0c-4a22-b33a-00b7daa48b76
  userName: Habsire
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjViNMotObNfrz6YyFDbLMN-nNz4Ek7KtNwHZlEqFd1xyUB-Lzw
  content: ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ቀን ሙከራ ራሱ አይሰራም በጣም የሚያስጠላ ሲስተም ነው እኔ ከአስር 1 እራሱ አልሰጠውም
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 1.9.14
  at: 2026-05-13 21:37:46
  replyContent: None
  repliedAt: None
  appVersion: 1.9.14


In [6]:
df = review_dataframe(reviews, app_info={'title': 'Dashen Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (700, 6)


,review_id,review,rating,date,bank,source
0,88c562ed-6c0c-4a22-b33a-00b7daa48b76,ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ...,1,2026-05-13 21:37:46,Dashen Bank,Google Play
1,abf6804b-429f-4383-94a1-5a5955c27569,bad mobile banking at all,1,2026-05-13 19:14:11,Dashen Bank,Google Play
2,265cfda4-296c-4676-bc76-032559a65ec2,very nice app.,5,2026-05-13 17:44:57,Dashen Bank,Google Play
3,94a4413f-40c7-4842-b31c-43f70aff0686,very difficult app,1,2026-05-13 16:26:31,Dashen Bank,Google Play
4,eaeb3ab3-2f40-49c2-be8f-8d434331b7cb,"Good app, but debit transactions not allowed W...",5,2026-05-13 11:41:27,Dashen Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [7]:
df_clean = df.copy()

In [8]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 700 reviews


#### Handle missing values

In [9]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 700 reviews


#### Normalize dates to YYYY-MM-DD format

In [10]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-13 21:37:46
1   2026-05-13 19:14:11
2   2026-05-13 17:44:57
dtype: datetime64[us]

After normalization:
0    2026-05-13
1    2026-05-13
2    2026-05-13
dtype: str

Date range: 2025-04-22 to 2026-05-13


#### Handle incorrect ratings

In [11]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 700 reviews


#### Clean review text

In [12]:
df['review'] = df['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df['review'].head(10).to_string())

Sample cleaned reviews:
0                                                  1  
1                            bad mobile banking at all
2                                       very nice app.
3                                   very difficult app
4    good app, but debit transactions not allowed w...
5                                                 nice
6    very bad customer service line. they won't pic...
7                                            smart app
8             the application booting time is so bad..
9                                                 good


#### Save the cleaned dataset

In [13]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (700, 5)


,review,rating,date,bank,source
0,ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ...,1,2026-05-13,Dashen Bank,Google Play
1,"Good app, but debit transactions not allowed W...",5,2026-05-13,Dashen Bank,Google Play
2,nice,5,2026-05-13,Dashen Bank,Google Play
3,bad mobile banking at all,1,2026-05-13,Dashen Bank,Google Play
4,very difficult app,1,2026-05-13,Dashen Bank,Google Play
5,very nice app.,5,2026-05-13,Dashen Bank,Google Play
6,Very bad customer service line. they won't pic...,1,2026-05-12,Dashen Bank,Google Play
7,smart app,5,2026-05-12,Dashen Bank,Google Play
8,The application booting time is so bad..,3,2026-05-12,Dashen Bank,Google Play
9,good,5,2026-05-12,Dashen Bank,Google Play


In [14]:
save_cleaned_data(df_clean, output_path="../data/processed/boa_reviews_cleaned.csv")

Cleaned data saved to ../data/processed/boa_reviews_cleaned.csv
Saved to: ../data/processed/boa_reviews_cleaned.csv


### Report

In [15]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    700
  Reviews after cleaning :    700
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-04-22  to  2026-05-13
Rating distribution:
  5 stars:  468  █████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:   47  █████████
  3 stars:   35  ███████
  2 stars:   26  █████
  1 stars:  124  ████████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 22 characters
    Max    : 499 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

